# VCPA-Inject (Generation 12) — Kaggle test notebook

This notebook runs the **complete** VCPA-Inject test and benchmark suite on a GPU.

## Setup (one time)
1. Upload the repository folder as a Kaggle **Dataset** named `vcpa-main`
   (or clone your repo in the first code cell).
2. `Settings → Accelerator → GPU T4 x2` (or P100).
3. `Settings → Internet → On` (needed to download the model).

## What gets measured
- **Integrity**: per-parameter SHA-256 before / during / after injection.
- **Accuracy**: perplexity on wikitext-2 plus next-token divergence (top-1 agreement, cosine, KL).
- **Prefill**: latency and tokens/s across context lengths, baseline vs VCPA.
- **Decode**: tokens/s with a KV cache (VCPA falls back to exact attention here).
- **Memory**: peak CUDA memory, baseline vs VCPA.

Everything is written to `/kaggle/working/vcpa_results/*.json`.

In [ ]:
!pip install -q --upgrade transformers accelerate datasets
!pip install -q bitsandbytes   # only needed for --load-in-4bit
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
import os, sys, glob

candidates = ['/kaggle/input/vcpa-main', '/kaggle/input/vcpa', os.getcwd()]
ROOT = next((p for p in candidates if os.path.isdir(os.path.join(p, 'vcpa'))), None)
if ROOT is None:
    # optional: clone your repository here instead
    # !git clone https://github.com/<you>/VCPA-main.git /kaggle/working/VCPA-main
    ROOT = '/kaggle/working/VCPA-main'
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print('project root:', ROOT)
print(sorted(os.listdir(ROOT))[:20])

In [ ]:
# Fast offline unit tests: ~2 seconds, tiny random model, no downloads
!python -m unittest discover -s tests -p "test_*.py" -v

In [ ]:
# Full suite on a real checkpoint.
# Start with 0.5B, then try 1.5B / 3B. For 7B add --load-in-4bit.
!python kaggle/run_on_kaggle.py \
    --models Qwen/Qwen2.5-0.5B Qwen/Qwen2.5-1.5B \
    --seq-lens 512 1024 2048 4096 \
    --memory-seq-lens 1024 2048 4096 8192 \
    --prompt-len 1024 --new-tokens 64 \
    --chunk-size 64 --local-window 256 --threshold 256

In [ ]:
# Optional sweep: find a (chunk_size, local_window) trade-off on your data
import subprocess, sys, json, os
configs = [(64, 128), (64, 256), (64, 512), (32, 128), (128, 256), (128, 512)]
results = {}
for chunk, window in configs:
    out = f'/kaggle/working/vcpa_results/sweep_c{chunk}_w{window}.json'
    subprocess.run([sys.executable, 'benchmarks/benchmark_accuracy.py',
                    '--model', 'Qwen/Qwen2.5-0.5B', '--dtype', 'float16',
                    '--chunk-size', str(chunk), '--local-window', str(window),
                    '--threshold', str(window), '--seq-lens', '2048',
                    '--max-windows', '8', '--json', out], check=False)
    try:
        results[(chunk, window)] = json.load(open(out))['rows'][0]
    except Exception as e:
        print('failed', chunk, window, e)

for (c, w), r in results.items():
    print(f'C={c:>3} W={w:>3}  ppl_delta={r["ppl_delta_pct"]:+.2f}%  top1={r["top1_agreement_pct"]:.1f}%  cos={r["mean_cosine_similarity"]:.4f}')

## How to read the results

- `ppl_delta_pct` is the honest quality cost of the approximation. Small (< ~2%) on
  wikitext is a good sign; large values mean the chunk/window pair is too aggressive.
- `top1_agreement_pct` near 100% means the model would usually pick the same next token.
- `speedup_cepl_vs_baseline > 1` only appears once the context is long enough for the
  O(N²) baseline to dominate; short contexts are intentionally routed to exact SDPA.
- `decode` ratio should be ≈ 1.0: VCPA does **not** approximate during decoding.
- `parameters_unchanged: True` in every JSON is the non-negotiable integrity guarantee.

## Known limitations
- CEPL is an approximation; it is never claimed to be equivalent to exact attention.
- Padded batches and any cache-bearing forward use the exact fallback.
- Benchmarks are hardware-dependent: compare baseline vs injected on the same GPU/session.
- The tiny-model unit tests prove mechanics, not language quality.